![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [62]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Load the dataset 
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


# PART 1 : Explore the Dataset

In [63]:
print(cc_apps.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       690 non-null    object 
 1   1       690 non-null    object 
 2   2       690 non-null    float64
 3   3       690 non-null    object 
 4   4       690 non-null    object 
 5   5       690 non-null    object 
 6   6       690 non-null    object 
 7   7       690 non-null    float64
 8   8       690 non-null    object 
 9   9       690 non-null    object 
 10  10      690 non-null    int64  
 11  11      690 non-null    object 
 12  12      690 non-null    int64  
 13  13      690 non-null    object 
dtypes: float64(2), int64(2), object(10)
memory usage: 75.6+ KB
None


In [64]:
print(cc_apps.isna().sum())

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
dtype: int64


In [65]:
for column in cc_apps.columns :
    print(cc_apps[column].value_counts())

b    468
a    210
?     12
Name: 0, dtype: int64
?        12
22.67     9
20.42     7
18.83     6
24.50     6
         ..
48.25     1
28.33     1
18.75     1
18.50     1
36.42     1
Name: 1, Length: 350, dtype: int64
1.500     21
0.000     19
3.000     19
2.500     19
0.750     16
          ..
0.085      1
12.250     1
11.045     1
11.125     1
3.375      1
Name: 2, Length: 215, dtype: int64
u    519
y    163
?      6
l      2
Name: 3, dtype: int64
g     519
p     163
?       6
gg      2
Name: 4, dtype: int64
c     137
q      78
w      64
i      59
aa     54
ff     53
k      51
cc     41
m      38
x      38
d      30
e      25
j      10
?       9
r       3
Name: 5, dtype: int64
v     399
h     138
bb     59
ff     57
?       9
j       8
z       8
dd      6
n       4
o       2
Name: 6, dtype: int64
0.000    70
0.250    35
0.040    33
1.000    31
0.125    30
         ..
4.165     1
9.000     1
1.960     1
5.125     1
8.290     1
Name: 7, Length: 132, dtype: int64
t    361
f    329
Name: 8

The symbol "?" in the data represents missing values, we're going to find a solution to address them

# PART 2 : Addressing missing data

In [66]:
threshold = len(cc_apps) * 0.05
print(threshold)

34.5


In [67]:
cols_to_drop = []
for column in cc_apps.columns:
    missing_count = (cc_apps[column] == "?").sum()
    if missing_count <= threshold and missing_count != 0 :
        cols_to_drop.append(column)

print(cols_to_drop)

[0, 1, 3, 4, 5, 6]


The total of missing values by column taken individually is less or equal to the threshold, so we can simply drop this missing values

In [68]:
cc_apps.replace("?", np.nan, inplace=True)
cc_apps.dropna(subset=cols_to_drop, inplace=True)

In [69]:
for column in cc_apps.columns:
    missing_count = (cc_apps[column] == "?").sum()
    print(f"{column} : {missing_count}")

0 : 0
1 : 0
2 : 0
3 : 0
4 : 0
5 : 0
6 : 0
7 : 0
8 : 0
9 : 0
10 : 0
11 : 0
12 : 0
13 : 0


# PART 3 : Handling categorical data

In [70]:
print(cc_apps.select_dtypes(include="object").columns)

Int64Index([0, 1, 3, 4, 5, 6, 8, 9, 11, 13], dtype='int64')


The column "1" contains numeric values (float) but the data type is object, we should correct that

In [71]:
cc_apps[1] = cc_apps[1].astype(float)
print(cc_apps[1].dtype)

float64


In [72]:
encoders = {} 

for col in cc_apps.select_dtypes(include="object").columns:
    le = LabelEncoder()
    cc_apps[col] = le.fit_transform(cc_apps[col])
    encoders[col] = le

print(cc_apps.head(5))

   0      1      2   3   4   5   6     7   8   9   10  11   12  13
0   1  30.83  0.000   1   0  12   7  1.25   1   1   1   0    0   0
1   0  58.67  4.460   1   0  10   3  3.04   1   1   6   0  560   0
2   0  24.50  0.500   1   0  10   3  1.50   1   0   0   0  824   0
3   1  27.83  1.540   1   0  12   7  3.75   1   1   5   0    3   0
4   1  20.17  5.625   1   0  12   7  1.71   1   0   0   2    0   0


In [73]:
print(cc_apps.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 659 entries, 0 to 689
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       659 non-null    int64  
 1   1       659 non-null    float64
 2   2       659 non-null    float64
 3   3       659 non-null    int64  
 4   4       659 non-null    int64  
 5   5       659 non-null    int64  
 6   6       659 non-null    int64  
 7   7       659 non-null    float64
 8   8       659 non-null    int64  
 9   9       659 non-null    int64  
 10  10      659 non-null    int64  
 11  11      659 non-null    int64  
 12  12      659 non-null    int64  
 13  13      659 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 77.2 KB
None


# PART 4 : Prepare Data for Modeling

In [74]:
data = cc_apps.drop(13, axis=1).values
target = cc_apps[13].values
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, 
random_state=42) 

In [75]:
steps = [ ('scaler', StandardScaler()),
          ('logreg', LogisticRegression())    
]
pipeline = Pipeline(steps)
params = {"logreg__solver": ["newton-cg", "saga", "lbfgs"],
         "logreg__C": np.linspace(0.001, 1.0, 10)}
cv =GridSearchCV(pipeline, param_grid=params, scoring='accuracy') 
cv.fit(X_train, y_train) 
y_pred = cv.predict(X_test) 

In [76]:
best_score = cv.best_score_
print(best_score) 

0.8728661275831087
